# FLUX Phase 1: Continuous Semantic Encoder (CSE)
## Train, Test & Demo — All-in-One Kaggle Notebook

> **Goal:** Replace the tokenizer + embedding layer with a continuous wave-based encoder that requires no vocabulary.

This notebook runs the complete Phase 1 pipeline:
1. Clone the repo & install dependencies
2. Train the CSE (reconstruction + semantic contrastive loss)
3. Run all 3 test scripts (reconstruction, language-agnostic, semantic proximity)
4. Run both demo scripts (wave visualization, interference patterns)
5. Save checkpoint & push results back to GitHub

**Hardware:** GPU recommended (P100/T4) but CPU works (~1hr vs ~10min)

---
## 1. Setup: Clone Repo & Install Dependencies

In [ ]:
# Clone the FLUX repo
!git clone https://github.com/Unseengap/FLUX.git
%cd FLUX

In [ ]:
# Install dependencies (most are pre-installed on Kaggle)
!pip install -q datasets rich faiss-cpu

In [ ]:
# Run project setup
!python setup.py

In [ ]:
# Verify imports & hardware
import torch
import sys

print(f"Python:  {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:     {torch.cuda.get_device_name(0)}")
    print(f"VRAM:    {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    DEVICE = 'cuda'
else:
    print("No GPU — using CPU (training will be slower)")
    DEVICE = 'cpu'

print(f"\nUsing device: {DEVICE}")

---
## 2. Quick Smoke Test
Verify the CSE builds, encodes, and gradients flow before training.

In [ ]:
import sys
from pathlib import Path

# Add paths
sys.path.insert(0, 'phases/phase1')
sys.path.insert(0, '.')

from phases.phase1.cse import ContinuousSemanticEncoder, WaveDecoder
from phases.phase1.wave_types import TOTAL_WAVE_DIM, WAVE_DIMS

# Build encoder + decoder
cse = ContinuousSemanticEncoder(device=DEVICE).to(DEVICE)
decoder = WaveDecoder(wave_dim=TOTAL_WAVE_DIM).to(DEVICE)

param_count = sum(p.numel() for p in cse.parameters())
dec_count = sum(p.numel() for p in decoder.parameters())
print(f"CSE parameters:     {param_count:,}")
print(f"Decoder parameters: {dec_count:,}")
print(f"Total parameters:   {param_count + dec_count:,}")

# Test encoding
test_inputs = ['Hello, world!', 'dog', '你好世界', 'def f(): pass', 'a', '']
for text in test_inputs:
    with torch.no_grad():
        wave = cse.encode(text)
    print(f'  "{text}" → [{wave.seq_len}, {wave.full.shape[-1]}]')

# Test gradient flow
wave = cse.encode('test backward pass')
logits = decoder(wave.full)
target = cse.text_to_bytes('test backward pass')
loss = torch.nn.functional.cross_entropy(logits[:len(target)], target)
loss.backward()
print(f'\n✓ Backward pass OK — loss = {loss.item():.4f}')
print('✓ All smoke tests passed!')

---
## 3. Train the CSE

Training objectives:
- **Reconstruction:** encode text → wave → decode back to bytes
- **Semantic contrastive:** similar words → similar waves, different words → different waves
- **Antonym separation:** opposite words → low similarity

**~5000 steps, ~10 min on GPU, ~60 min on CPU**

In [ ]:
%cd /kaggle/working/FLUX/phases/phase1
!python train_cse.py --steps 5000 --device {DEVICE}

In [ ]:
# Verify checkpoint was saved
import os
ckpt_path = '/kaggle/working/FLUX/checkpoints/phase1.phase.pt'
if os.path.exists(ckpt_path):
    size_mb = os.path.getsize(ckpt_path) / 1e6
    print(f'✓ Checkpoint saved: {ckpt_path} ({size_mb:.1f} MB)')
else:
    print('✗ Checkpoint NOT found — training may have failed')

---
## 4. Run Tests

### Test 1: Reconstruction Quality
- Average reconstruction loss < 0.1 (byte accuracy > 90%)
- No sentence has loss > 0.5

In [ ]:
%cd /kaggle/working/FLUX/phases/phase1
!python test_phase1_test1.py

### Test 2: Language-Agnostic Encoding
- Encodes any UTF-8 input without errors
- Cross-language similarity > 0.4
- Deterministic encoding

In [ ]:
!python test_phase1_test2.py

### Test 3: Semantic Proximity Ordering
- sim("dog", "cat") > sim("dog", "democracy")
- 18/20 triplets must pass

In [ ]:
!python test_phase1_test3.py

---
## 5. Run Demos

### Demo 1: Semantic Wave Visualization
Heatmap of wave components (phonetic, syntactic, semantic, temporal, intensity) for sample text.

In [ ]:
!python demo_phase1_demo1.py

In [ ]:
# Display the generated visualization
from IPython.display import Image, display
img_path = '/kaggle/working/FLUX/phases/phase1/demo1_wave_visualization.png'
if os.path.exists(img_path):
    display(Image(filename=img_path, width=900))
else:
    print('Visualization image not found')

### Demo 2: Interference Patterns
Shows constructive (similar words), destructive (opposite words), and neutral (unrelated) interference.

In [ ]:
!python demo_phase1_demo2.py

---
## 6. Interactive Exploration
Try encoding your own text and comparing similarities.

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/FLUX/phases/phase1')
sys.path.insert(0, '/kaggle/working/FLUX')

import torch
import torch.nn.functional as F
from cse import ContinuousSemanticEncoder
from flux_utils import load_checkpoint

# Load trained model
checkpoint = load_checkpoint(1)
cse = ContinuousSemanticEncoder(**checkpoint['config'], device='cpu')
cse.load_state_dict(checkpoint['state_dict'])
cse.eval()

def similarity(text1, text2):
    """Compute semantic similarity between two texts."""
    with torch.no_grad():
        w1 = cse.encode(text1)
        w2 = cse.encode(text2)
    v1 = w1.semantic.mean(dim=0)
    v2 = w2.semantic.mean(dim=0)
    return F.cosine_similarity(v1.unsqueeze(0), v2.unsqueeze(0)).item()

# Try your own pairs!
pairs = [
    ("dog", "cat"),
    ("dog", "democracy"),
    ("king", "queen"),
    ("hot", "cold"),
    ("hello", "bonjour"),
    ("python code", "javascript code"),
    ("I love pizza", "Pizza is my favorite food"),
]

print(f"{'Text 1':<30} {'Text 2':<30} {'Similarity':>10}")
print("-" * 72)
for t1, t2 in pairs:
    sim = similarity(t1, t2)
    print(f"{t1:<30} {t2:<30} {sim:>10.4f}")

---
## 7. View Results

In [ ]:
# Print the auto-generated results file
results_path = '/kaggle/working/FLUX/phases/phase1/RESULTS_PHASE_1.md'
if os.path.exists(results_path):
    from IPython.display import Markdown, display
    display(Markdown(open(results_path).read()))
else:
    print('Results file not yet generated — run tests first')

---
## 8. Save Artifacts
Copy checkpoint and results to Kaggle output for download.

In [ ]:
import shutil

output_dir = Path('/kaggle/working/output')
output_dir.mkdir(exist_ok=True)

# Copy checkpoint
ckpt_src = Path('/kaggle/working/FLUX/checkpoints/phase1.phase.pt')
if ckpt_src.exists():
    shutil.copy(ckpt_src, output_dir / 'phase1.phase.pt')
    print(f'✓ Checkpoint copied to output/')

# Copy results
for f in Path('/kaggle/working/FLUX/phases/phase1').glob('RESULTS_*'):
    shutil.copy(f, output_dir / f.name)
    print(f'✓ {f.name} copied to output/')

# Copy demo visualization
viz = Path('/kaggle/working/FLUX/phases/phase1/demo1_wave_visualization.png')
if viz.exists():
    shutil.copy(viz, output_dir / viz.name)
    print(f'✓ Visualization copied to output/')

print(f'\nOutput files:')
for f in sorted(output_dir.iterdir()):
    print(f'  {f.name} ({f.stat().st_size / 1e6:.1f} MB)')

---
## Phase 1 Summary

| Acceptance Criteria | Status |
|---|---|
| Any UTF-8 string encodes without errors | Run Test 2 |
| Similar words produce cosine similarity > 0.7 | Run Test 3 |
| Encoding is deterministic | Run Test 2 |
| Reconstruction loss < 0.1 | Run Test 1 |
| Encodes English, Chinese, code, math equally | Run Test 2 |
| All tests pass | Check above |
| Both demos run | Check above |
| Checkpoint saved | Check cell 4 |

**When all criteria pass → Phase 1 complete → Phase 2 unlocked!**